In [1]:
#Parameters
ticker = "AAPL"
start_date = "2024-06-01"
end_date = "2024-11-30"
db_path = "data/data.db"

In [2]:
# Parameters
ticker = "MSFT"
start_date = "2024-06-01"
end_date = "2024-11-30"
db_path = "C:\\Users\\shrav\\ucb_mfe\\UCB-MFE-python-preprogram-2026-winter\\Homeworks\\HW2\\Shravani_Koli\\data\\data.db"


In [3]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Connect to SQLite
conn = sqlite3.connect(db_path)
# Load cleaned stock data 
query = """
SELECT ts, open, high, low, close, adj_close, volume, ticker
FROM ohlc_hw_cleaned
WHERE ticker = ?
  AND ts >= ?
  AND ts <= ?
ORDER BY ts
"""

stock_df = pd.read_sql_query(query, conn, params=(ticker, start_date, end_date),parse_dates=["ts"])
conn.close()
print("Ticker:", ticker)
print("Number of records:", len(stock_df))
print("Date range:", start_date, "to", end_date)

stock_df.head()

Ticker: MSFT
Number of records: 92
Date range: 2024-06-01 to 2024-11-30


,ts,open,high,low,close,adj_close,volume,ticker
0,2024-06-03,415.53,416.43,408.92,413.52,412.78,17484675,MSFT
1,2024-06-04,412.43,416.44,409.68,416.07,415.32,14348917,MSFT
2,2024-06-05,417.81,424.08,416.30,424.01,423.25,16988038,MSFT
3,2024-06-06,424.01,425.31,420.58,424.52,423.76,14861251,MSFT
4,2024-06-07,426.20,426.28,423.00,423.85,423.09,13621650,MSFT


In [4]:
#daily logarithmic returns using adjusted closing prices
stock_df["log_return"] = np.log(stock_df["adj_close"] / stock_df["adj_close"].shift(1))

# Calculate 20-day rolling volatility, annualized
stock_df["rolling_volatility"] = (stock_df["log_return"].rolling(window=20).std()* np.sqrt(252))

#cumulative returns
stock_df["cumulative_return"] = (stock_df["adj_close"] / stock_df["adj_close"].iloc[0])
#running peak
stock_df["running_peak"] = stock_df["cumulative_return"].cummax()
#calculate drawdown
stock_df["drawdown"] = (stock_df["cumulative_return"] / stock_df["running_peak"] - 1)

# the calculated columns
stock_df[
    ["ts", "adj_close", "log_return",
     "rolling_volatility", "cumulative_return", "drawdown"]
].head(25)

,ts,adj_close,log_return,rolling_volatility,cumulative_return,drawdown
0,2024-06-03,412.78,NaN,NaN,1.000000,0.000000
1,2024-06-04,415.32,0.006135,NaN,1.006153,0.000000
2,2024-06-05,423.25,0.018914,NaN,1.025365,0.000000
3,2024-06-06,423.76,0.001204,NaN,1.026600,0.000000
4,2024-06-07,423.09,-0.001582,NaN,1.024977,-0.001581
5,2024-06-10,427.10,0.009433,NaN,1.034692,0.000000
6,2024-06-11,431.90,0.011176,NaN,1.046320,0.000000
7,2024-06-12,440.27,0.019194,NaN,1.066597,0.000000
8,2024-06-13,440.79,0.001180,NaN,1.067857,0.000000
9,2024-06-14,441.77,0.002221,NaN,1.070231,0.000000


In [5]:
# Using 2% annual risk-free rate
risk_free_rate = 0.02
trading_days = 252
daily_returns = stock_df["log_return"].dropna()

#performance metrics
total_return = (stock_df["adj_close"].iloc[-1] /stock_df["adj_close"].iloc[0] - 1)

annualized_return = np.exp(daily_returns.mean() * trading_days) - 1
annualized_volatility = (daily_returns.std() * np.sqrt(trading_days))

sharpe_ratio = ((daily_returns.mean() * trading_days - risk_free_rate)/ annualized_volatility if annualized_volatility > 0 else np.nan)

max_drawdown = stock_df["drawdown"].min()
best_day = np.exp(daily_returns.max()) - 1
worst_day = np.exp(daily_returns.min()) - 1
positive_days = (daily_returns > 0).mean() * 100

avg_up_day = (np.exp(daily_returns[daily_returns > 0]) - 1).mean()

avg_down_day = (np.exp(daily_returns[daily_returns < 0]) - 1).mean()

#displaying the 10 metrics 
metrics = pd.DataFrame({
    "Metric": [
        "Total Return",
        "Annualized Return",
        "Annualized Volatility",
        "Sharpe Ratio",
        "Maximum Drawdown",
        "Best Day",
        "Worst Day",
        "Positive Days (%)",
        "Average Up Day",
        "Average Down Day"
    ],
    "Value": [
        total_return,
        annualized_return,
        annualized_volatility,
        sharpe_ratio,
        max_drawdown,
        best_day,
        worst_day,
        positive_days,
        avg_up_day,
        avg_down_day
    ]
})

metrics

,Metric,Value
0,Total Return,0.013034
1,Annualized Return,0.036511
2,Annualized Volatility,14.366684
3,Sharpe Ratio,0.001104
4,Maximum Drawdown,-0.955321
5,Best Day,21.606528
6,Worst Day,-0.954922
7,Positive Days (%),53.846154
8,Average Up Day,1.465877
9,Average Down Day,-0.118549


In [6]:
import scrapbook as sb

# Store the metrics so Papermill and Scrapbook can retrieve them
report_metrics = {
    "ticker": ticker,
    "avg_daily_return": float(daily_returns.mean()),
    "avg_volatility": float(stock_df["rolling_volatility"].mean()),
    "max_drawdown": float(max_drawdown),
    "sharpe_ratio": float(sharpe_ratio),
    "total_return": float(total_return)
}

sb.glue("report_metrics", report_metrics)

print("Report metrics saved for:", ticker)
print(report_metrics)

Report metrics saved for: MSFT
{'ticker': 'MSFT', 'avg_daily_return': 0.00014230078047734358, 'avg_volatility': 13.291131473541203, 'max_drawdown': -0.9553212056300089, 'sharpe_ratio': 0.0011039288311260973, 'total_return': 0.0130335772081982}


In [7]:
#finding dates with the largest daily price movements
stock_df["daily_return"] = stock_df["adj_close"].pct_change()

extreme_days = stock_df.loc[stock_df["daily_return"].abs().nlargest(10).index,["ts", "adj_close", "daily_return"]]
extreme_days

,ts,adj_close,daily_return
45,2024-08-08,8990.842427,21.606528
42,2024-08-05,6094.235773,13.945644
74,2024-09-24,5821.411112,12.428551
30,2024-07-18,5714.914976,11.908644
79,2024-10-01,5400.806850,11.551259
46,2024-08-09,405.290000,-0.954922
43,2024-08-06,398.890000,-0.934546
75,2024-09-25,432.110000,-0.925772
31,2024-07-19,436.320000,-0.923652
80,2024-10-02,417.130000,-0.922765


In [8]:
#inspecting suspicious AAPL price records
stock_df.loc[
    stock_df["ts"].isin(
        pd.to_datetime([
            "2024-06-27",
            "2024-08-22",
            "2024-09-17"
        ])
    ),
    ["ts", "open", "high", "low", "close", "adj_close"]
]

,ts,open,high,low,close,adj_close
17,2024-06-27,452.18,456.17,451.77,452.85,452.04
54,2024-08-22,424.36,426.79,414.61,415.55,415.55
69,2024-09-17,440.23,441.85,432.27,435.15,435.15
